In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [7]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its details."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available products: {', '.join(PRODUCTS)}"
    return str(p)

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful assistant for an online tech store."
)

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role":"user", "content": question}]})
    print(result["messages"][-1].content)

In [8]:
ask("What is the price of a wireless headphones?")

The wireless headphones are priced at **$79.99**.

Here are a few more details about them:
*   **Rating:** 4.6 stars
*   **Features:** Over-ear Bluetooth design, 30-hour battery life, and active noise cancellation.


In [9]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [10]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful assistant for an online tech store."
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role":"user", "content": question}]})
    print(result["messages"][-1].content)

In [11]:
ask2("How do people like smart watch?")

Based on the available data, **smart watches** have an average rating of **3.9 out of 5**, based on **340 reviews**.

This suggests that customers generally have a positive experience with smart watches, though there’s still room for improvement (as a 3.9 rating indicates some users may have had mixed or less-than-ideal experiences). Common themes in smart watch reviews often include praise for fitness tracking, notifications, and design, while criticisms may involve battery life, app compatibility, or price.

Would you like details on a specific smart watch model?


# Agent with MEMORY

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

agent3 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask3(question: str):
    config = {"configurable": {"thread_id": "specific-user-session"}}
    result = agent3.invoke(
        {"messages": [{"role":"user", "content": question}]},
        config=config
    )
    print(result["messages"][-1].content)

In [15]:
ask3("What is the price of a wireless headphones?")

The wireless headphones are priced at **$79.99**.

Here are a few more details about them:
- **Rating:** 4.6 stars
- **Features:** Over-ear Bluetooth design, 30-hour battery life, and active noise cancellation.


In [16]:
ask3("What are the reviews for this product?")

The wireless headphones have an average rating of **4.6 stars** based on **1,262 reviews**.

Unfortunately, the specific text of individual customer reviews isn't available in the data I can access, but the high rating and large number of reviews suggest they are generally well-received by customers.
